# MLOps：視聴データからF1層の有無を予測する

5局の合成視聴データを観察し、XGBoostを学習・評価します。Model Registryへ登録した同じモデルで、判断理由の説明、20,000台の予測、結果保存までを行います。

モデル本体、版、評価指標、実行環境、予測関数、説明関数をSnowflake上でまとめて管理できることを確認します。Markdown説明の下にあるPythonコードセルを上から1つずつ実行してください。

## 1. Notebookを準備する

**このセルで行うこと**

- Snowflakeのセッションを取得する
- 使用するロール・ウェアハウス・データベースを設定する
- pandas、scikit-learn、XGBoost、Snowpark、Snowflake MLを読み込む
- 登録するモデル名 `TV_F1_PRESENCE_MODEL` と版 `V3` を設定する

この章で使うモデルは **XGBoost** です。複数の決定木を組み合わせ、F1層がいる確率を0〜1で計算する分類モデルです。

**実行**

次のコードセルを実行します。

**成功の目印**

`パッケージ: scikit-learn ... / XGBoost ...` と `準備完了: TV_F1_PRESENCE_MODEL V3` が表示されます。

In [ ]:
for state_name in ('config_token', 'data_token', 'evaluation_token', 'registration_token', 'prediction_token', 'publication_token', 'model', 'registered_version', 'prediction_pdf'):
    globals().pop(state_name, None)
import re
import json
import hashlib
import uuid
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import sklearn
import xgboost
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.inspection import permutation_importance
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as sf
from snowflake.snowpark.types import StructType, StructField, StringType, LongType, DoubleType
from snowflake.ml.registry import Registry
from snowflake.ml.model import model_signature

session = get_active_session()
if session.get_current_role().strip('"') != 'BCAST_PLATFORM_ENGINEER_ROLE':
    raise RuntimeError('NotebookのロールをBCAST_PLATFORM_ENGINEER_ROLEに変更してください。')
session.use_warehouse('BCAST_PLATFORM_COMMON_WH')
session.use_database('BCAST_PLATFORM_HANDSON')
session.use_schema('ML')
if tuple(map(int, sklearn.__version__.split('.')[:2])) < (1, 4):
    raise RuntimeError('このNotebookにはscikit-learn 1.4以上が必要です。')
if tuple(map(int, xgboost.__version__.split('.')[:2])) < (2, 0):
    raise RuntimeError('このNotebookにはXGBoost 2.0以上が必要です。')
print('パッケージ:', 'scikit-learn', sklearn.__version__, '/ XGBoost', xgboost.__version__)
MODEL_NAME = 'TV_F1_PRESENCE_MODEL'
MODEL_VERSION = 'V3'
DATASET_VERSION = 'F1_SIGNAL_V2'
THRESHOLD = 0.50
SPLIT_SEED = 20260918
EVALUATION_SCHEMA_VERSION = 'XGBOOST_V1'
EXPECTED_DEVICES = {f'C{number:06d}' for number in range(1, 20001)}
GENRES = ['NEWS', 'DRAMA', 'VARIETY', 'ANIME', 'SPORTS', 'MUSIC', 'MOVIE', 'INFO']
FEATURE_COLUMNS = [genre + '_SHARE' for genre in GENRES] + ['TOTAL_MINUTES', 'TOTAL_SESSIONS', 'ACTIVE_DAYS', 'MEAN_MINUTES']
RELEASE_TABLES = {f'VIEWING_LOG_NW{number:02d}': ['EVENT_ID', 'NETWORK_ID', 'DEVICE_ID', 'VIEW_FROM', 'VIEW_TO', 'GENRE'] for number in range(1, 6)}
RELEASE_TABLES.update(DEVICE_LABELS=['DEVICE_ID', 'LABEL_AVAILABLE', 'TARGET_F1'], PROGRAM_MASTER=['PROGRAM_ID', 'PROGRAM_NAME', 'NETWORK_ID', 'GENRE', 'TIME_SLOT', 'DURATION_MIN', 'SYNOPSIS'], PROGRAM_SCHEDULE=['PROGRAM_ID', 'NETWORK_ID', 'AIR_DATE', 'AIR_FROM', 'AIR_TO'])
RELEASE_COUNTS = dict(zip(RELEASE_TABLES, [195938, 184465, 179991, 234324, 255930, 20000, 60, 8747]))

def invalidate_from(stage):
    stages = ['data', 'evaluation', 'registration', 'prediction', 'publication']
    values = {'data': ['features_pdf', 'known_pdf'], 'evaluation': ['model', 'metrics_df', 'final_test_record'], 'registration': ['registered_version'], 'prediction': ['prediction_pdf'], 'publication': ['published_table']}
    for downstream in stages[stages.index(stage):]:
        globals().pop(downstream + '_token', None)
        for key in values[downstream]:
            globals().pop(key, None)

def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':'), allow_nan=False).encode()).hexdigest()

def frame_digest(frame, columns):
    ordered = frame[columns].sort_values(columns[0]).reset_index(drop=True)
    return hashlib.sha256(ordered.to_csv(index=False, float_format='%.17g', na_rep='NULL', lineterminator='\n').encode()).hexdigest()

def configuration():
    return digest([MODEL_NAME, MODEL_VERSION, DATASET_VERSION, THRESHOLD, SPLIT_SEED, EVALUATION_SCHEMA_VERSION, FEATURE_COLUMNS, sklearn.__version__, xgboost.__version__])

def require_config():
    if SPLIT_SEED != 20260918:
        raise ValueError('この教材の分割seedは20260918で固定です。最終テストの再分割は行いません。')
    if globals().get('config_token') != configuration():
        raise ValueError('設定が変わりました。先頭から実行してください。')

def read_release():
    snapshot = []
    for table_name, columns in RELEASE_TABLES.items():
        columns_sql = ', '.join(columns)
        actual = session.sql(f'SELECT COUNT(*) AS ROW_COUNT, HASH_AGG({columns_sql}) AS ROW_FINGERPRINT FROM BCAST_PLATFORM_HANDSON.RAW.{table_name}').to_pandas()
        if len(actual) != 1 or actual[['ROW_COUNT', 'ROW_FINGERPRINT']].isna().any().any():
            raise ValueError('RAWの件数・照合値を取得できません。第1章を確認してください。')
        row_count = int(actual['ROW_COUNT'].iloc[0])
        row_fingerprint = int(actual['ROW_FINGERPRINT'].iloc[0])
        if row_count != RELEASE_COUNTS[table_name]:
            raise ValueError('RAWの件数が教材の期待件数と一致しません。第1章を確認してください。')
        snapshot.append([table_name, row_count, row_fingerprint])
    return digest(snapshot)

def collect_features(daily_source):
    result = daily_source.group_by('DEVICE_ID').agg(
        sf.sum('VIEW_MINUTES').cast('double').alias('TOTAL_MINUTES'),
        sf.sum('SESSION_COUNT').cast('double').alias('TOTAL_SESSIONS'),
        sf.count_distinct('VIEW_DATE').cast('double').alias('ACTIVE_DAYS'),
        *[sf.sum(sf.when(sf.col('GENRE') == genre, sf.col('VIEW_MINUTES')).otherwise(sf.lit(0))).cast('double').alias(genre + '_MINUTES') for genre in GENRES]
    )
    for genre in GENRES:
        result = result.with_column(genre + '_SHARE', sf.col(genre + '_MINUTES') / sf.nullif(sf.col('TOTAL_MINUTES'), sf.lit(0)))
    result = result.with_column('MEAN_MINUTES', sf.col('TOTAL_MINUTES') / sf.nullif(sf.col('TOTAL_SESSIONS'), sf.lit(0)))
    frame = result.select('DEVICE_ID', *FEATURE_COLUMNS).sort('DEVICE_ID').limit(20001).to_pandas()
    frame[FEATURE_COLUMNS] = frame[FEATURE_COLUMNS].astype('float64').round(8)
    if len(frame) != 20000 or frame.DEVICE_ID.isna().any() or not frame.DEVICE_ID.is_unique or set(frame.DEVICE_ID) != EXPECTED_DEVICES:
        raise ValueError('全20000台のIDが一致しません。')
    if not np.isfinite(frame[FEATURE_COLUMNS].to_numpy()).all() or (frame[FEATURE_COLUMNS] < 0).any().any():
        raise ValueError('特徴量に欠損・無限大・負数があります。')
    if not np.allclose(frame[FEATURE_COLUMNS[:8]].sum(axis=1), 1, rtol=0, atol=5e-8):
        raise ValueError('ジャンル割合の合計が1ではありません。')
    return frame

def current_data_token():
    require_config()
    current_fingerprint = read_release()
    if current_fingerprint != release_fingerprint:
        raise ValueError('RAWが取得開始時から変わりました。先頭から実行してください。')
    return digest([configuration(), current_fingerprint, frame_digest(features_pdf, ['DEVICE_ID'] + FEATURE_COLUMNS), frame_digest(known_pdf, ['DEVICE_ID'] + FEATURE_COLUMNS + ['TARGET_F1'])])

def require_data():
    if known_pdf.DEVICE_ID.tolist() != sorted(known_pdf.DEVICE_ID):
        raise ValueError('公開正解のDEVICE_ID順が変わりました。先頭から実行してください。')
    if globals().get('data_token') != current_data_token():
        raise ValueError('リリース・特徴量・公開正解が変わりました。先頭から実行してください。')
    live = collect_features(session.table('BCAST_PLATFORM_HANDSON.COMMON.VIEWING_DAILY'))
    if live.DEVICE_ID.tolist() != features_pdf.DEVICE_ID.tolist() or not np.allclose(live[FEATURE_COLUMNS], features_pdf[FEATURE_COLUMNS], rtol=0, atol=1.01e-8):
        raise ValueError('共通マートが変わりました。先頭から実行してください。')

def canonical_parameter(value):
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, (float, np.floating)):
        if np.isnan(value):
            return 'NaN'
        if np.isinf(value):
            return 'Infinity' if value > 0 else '-Infinity'
        return float(value)
    return value

def model_digest(estimator):
    parameters = {
        key: canonical_parameter(value)
        for key, value in estimator.get_params(deep=True).items()
        if value is None or isinstance(value, (str, int, float, bool, np.integer, np.floating))
    }
    booster_bytes = bytes(estimator.get_booster().save_raw())
    return digest([
        type(estimator).__module__,
        type(estimator).__name__,
        parameters,
        estimator.classes_.tolist(),
        estimator.feature_names_in_.tolist(),
        hashlib.sha256(booster_bytes).hexdigest(),
    ])

def metric_records(frame):
    columns = [
        'SPLIT',
        'MODEL',
        'ROC_AUC',
        'AVERAGE_PRECISION',
        'BRIER',
        'PRECISION',
        'RECALL',
        'F1_SCORE',
        'PREVALENCE',
        'CONFUSION',
    ]
    if list(frame.columns) != columns or frame[columns[:-1]].isna().any().any():
        raise ValueError('評価指標の列または値が不正です。')
    records = frame[columns].to_dict(orient='records')
    for record in records:
        numeric_values = [record[column] for column in columns[2:-1]]
        confusion = np.asarray(record['CONFUSION'], dtype='float64')
        if (
            not np.isfinite(np.asarray(numeric_values, dtype='float64')).all()
            or confusion.shape != (2, 2)
            or not np.isfinite(confusion).all()
        ):
            raise ValueError('評価指標に欠損または無限大があります。')
    return records

def evaluated_state():
    return digest([data_token, configuration(), model_digest(model), metric_records(metrics_df), *[frame_digest(part, ['DEVICE_ID'] + FEATURE_COLUMNS + ['TARGET_F1']) for part in (train_pdf, validation_pdf, test_pdf)]])

def require_evaluation():
    require_data()
    if globals().get('evaluation_token') != evaluated_state():
        raise ValueError('評価したモデル・分割と一致しません。先頭から実行してください。')

def require_registration():
    require_evaluation()
    if globals().get('registration_token') != digest([evaluation_token, registered_name, registered_version_name]) or (registered_name, registered_version_name) != (MODEL_NAME, MODEL_VERSION) or (registered_version.model_name, registered_version.version_name) != (MODEL_NAME, MODEL_VERSION):
        raise ValueError('登録の来歴が一致しません。先頭から実行してください。')

if not re.fullmatch(r'V[2-9]|V[1-9][0-9]+', MODEL_VERSION):
    raise ValueError('MODEL_VERSIONはV2、V3などを指定します。V1は上書きしません。')
config_token = configuration()
print('準備完了:', MODEL_NAME, MODEL_VERSION)

## 2. 予測に使うデータを準備する

**入力**

第2章で作成した5局共通の日次視聴データを使います。

**このセルで行うこと**

テレビ1台につき1行へ集計し、モデルへ渡す12項目を作ります。

- 8ジャンルの視聴割合
- 総視聴時間
- 視聴回数
- 視聴した日数
- 1回あたりの平均視聴時間

20,000台のうち、正解ラベルがある2,000台を学習と評価に使います。

**実行**

次のコードセルを実行します。

**成功の目印**

`データ準備完了: 全体 20000台 / 学習・評価対象 2000台` と、先頭5行が表示されます。

In [ ]:
invalidate_from('data')
require_config()
release_fingerprint = read_release()
features_pdf = collect_features(session.table('BCAST_PLATFORM_HANDSON.COMMON.VIEWING_DAILY'))
normalization = ' '.join(f"WHEN '{''.join(chr(ord(character) + 0xFEE0) for character in genre)}' THEN '{genre}'" for genre in GENRES)
station_ctes = []
for number in range(1, 6):
    station_ctes.append(f"""clean_nw{number:02d} AS (
 SELECT DISTINCT EVENT_ID, NETWORK_ID, DEVICE_ID, VIEW_FROM, VIEW_TO, GENRE
 FROM BCAST_PLATFORM_HANDSON.RAW.VIEWING_LOG_NW{number:02d} WHERE VIEW_TO > VIEW_FROM
 AND DATEDIFF('nanosecond', VIEW_FROM, VIEW_TO) <= 86400000000000
), normalized_nw{number:02d} AS (
 SELECT NETWORK_ID, DEVICE_ID, VIEW_FROM::DATE AS VIEW_DATE,
 CASE UPPER(TRIM(GENRE, ' \t\r\n　')) {normalization}
 ELSE UPPER(TRIM(GENRE, ' \t\r\n　')) END AS GENRE,
 DATEDIFF('nanosecond', VIEW_FROM, VIEW_TO)::FLOAT / 60000000000.0 AS VIEW_MINUTES
 FROM clean_nw{number:02d}
), daily_nw{number:02d} AS (
 SELECT NETWORK_ID, DEVICE_ID, VIEW_DATE, GENRE, COUNT(*) AS SESSION_COUNT,
  SUM(VIEW_MINUTES) AS VIEW_MINUTES
 FROM normalized_nw{number:02d} GROUP BY NETWORK_ID, DEVICE_ID, VIEW_DATE, GENRE
)""")
daily_union = ' UNION ALL '.join(f'SELECT NETWORK_ID, DEVICE_ID, VIEW_DATE, GENRE, SESSION_COUNT, VIEW_MINUTES FROM daily_nw{number:02d}' for number in range(1, 6))
source_daily = session.sql('WITH ' + ',\n'.join(station_ctes) + '\n' + daily_union)
source_features = collect_features(source_daily)
if features_pdf.DEVICE_ID.tolist() != source_features.DEVICE_ID.tolist() or not np.allclose(features_pdf[FEATURE_COLUMNS], source_features[FEATURE_COLUMNS], rtol=0, atol=1.01e-8):
    raise ValueError('共通マートとV2 RAWの特徴量が一致しません。第2章の全dbt buildを再実行してください。')
labels_pdf = session.table('BCAST_PLATFORM_HANDSON.RAW.DEVICE_LABELS').select('DEVICE_ID', 'LABEL_AVAILABLE', 'TARGET_F1').sort('DEVICE_ID').limit(20001).to_pandas()
if len(labels_pdf) != 20000 or labels_pdf.DEVICE_ID.isna().any() or not labels_pdf.DEVICE_ID.is_unique or set(labels_pdf.DEVICE_ID) != EXPECTED_DEVICES:
    raise ValueError('公開正解の全20000台のIDが一致しません。')
if labels_pdf.LABEL_AVAILABLE.isna().any() or not labels_pdf.LABEL_AVAILABLE.isin([True, False]).all():
    raise ValueError('正解公開フラグが不正です。')
known_mask = labels_pdf.LABEL_AVAILABLE.eq(True)
if known_mask.sum() != 2000 or labels_pdf.loc[~known_mask, 'TARGET_F1'].notna().any():
    raise ValueError('正解公開2000台・不明18000台の契約に一致しません。')
known_pdf = features_pdf.merge(labels_pdf.loc[known_mask, ['DEVICE_ID', 'TARGET_F1']], on='DEVICE_ID', validate='one_to_one').sort_values('DEVICE_ID').reset_index(drop=True)
if known_pdf.TARGET_F1.isna().any() or not known_pdf.TARGET_F1.isin([0, 1]).all():
    raise ValueError('公開正解は0/1が必要です。NULLは不明であり0ではありません。')
if known_pdf.TARGET_F1.value_counts().to_dict() != {0: 1656, 1: 344}:
    raise ValueError('V2公開正解のクラス件数が一致しません。')
if read_release() != release_fingerprint:
    raise ValueError('取得中にリリースが変わりました。先頭から実行してください。')
data_token = current_data_token()
print('データ準備完了: 全体', len(features_pdf), '台 / 学習・評価対象', len(known_pdf), '台')
features_pdf.head()

## 3. 学習データを観察する（EDA）

機械学習を始める前に、学習に使う1,200台のデータを観察します。これをEDA（探索的データ分析）と呼びます。

**このセルで見ること**

- F1層なし・ありの台数
- 2つのグループで視聴傾向に違いがあるか

**実行**

次のコードセルを実行します。

**成功の目印**

1,200台の内訳と、主な視聴項目のグループ別平均が表示されます。

In [ ]:
# EDA用に、正解が分かっている2,000台から学習用1,200台を分ける
remaining_indices, _ = train_test_split(
    np.arange(len(known_pdf)),
    test_size=0.2,
    random_state=SPLIT_SEED,
    stratify=known_pdf.TARGET_F1,
)
eda_indices, _ = train_test_split(
    remaining_indices,
    test_size=0.25,
    random_state=SPLIT_SEED,
    stratify=known_pdf.TARGET_F1.iloc[remaining_indices],
)
eda_pdf = known_pdf.iloc[eda_indices].copy()

# 正解ラベルの内訳を確認
label_counts = (
    eda_pdf["TARGET_F1"]
    .map({0: "F1層なし", 1: "F1層あり"})
    .value_counts()
    .rename_axis("GROUP")
    .reset_index(name="TV_COUNT")
)

# 分かりやすい5項目についてグループ別の平均を比較
eda_columns = ["DRAMA_SHARE", "VARIETY_SHARE", "ANIME_SHARE", "TOTAL_MINUTES", "ACTIVE_DAYS"]
group_means = (
    eda_pdf
    .assign(GROUP=eda_pdf["TARGET_F1"].map({0: "F1層なし", 1: "F1層あり"}))
    .groupby("GROUP")[eda_columns]
    .mean()
    .round(3)
)

print("学習データ1,200台の内訳")
display(label_counts)
print("グループ別の平均")
display(group_means)

## 4. ジャンル別の視聴傾向をグラフで見る

学習に使う1,200台について、F1層なし・ありの2グループで8ジャンルの平均視聴割合を比べます。

棒の高さに違いがあれば、その視聴傾向をモデルが予測の手がかりとして学習できます。モデルは1項目だけで判断せず、12項目を組み合わせます。

**実行**

次のコードセルを実行します。

**成功の目印**

2グループのジャンル別平均を並べた棒グラフが表示されます。

In [ ]:
import plotly.express as px

# 学習データ1,200台の8ジャンル平均をグラフ用の縦長データへ変換
genre_columns = [genre + "_SHARE" for genre in GENRES]
genre_means = (
    eda_pdf
    .assign(GROUP=eda_pdf["TARGET_F1"].map({0: "F1層なし", 1: "F1層あり"}))
    .groupby("GROUP")[genre_columns]
    .mean()
    .reset_index()
    .melt(id_vars="GROUP", var_name="GENRE", value_name="AVERAGE_SHARE")
)
genre_means["GENRE"] = genre_means["GENRE"].str.replace("_SHARE", "", regex=False)

fig = px.bar(
    genre_means,
    x="GENRE",
    y="AVERAGE_SHARE",
    color="GROUP",
    barmode="group",
    title="学習データのジャンル別平均視聴割合",
    labels={"GENRE": "ジャンル", "AVERAGE_SHARE": "平均視聴割合", "GROUP": "グループ"},
    template="plotly_white",
)
fig.update_layout(yaxis_tickformat=".0%")
fig.show()

## 5. XGBoostモデルを学習・評価する

ここから機械学習を行います。

**入力**

- 1台につき12項目の視聴傾向
- 正解ラベル（F1層なし=0、あり=1）

**このセルで行うこと**

- 2,000台を学習用1,200台、調整用400台、評価用400台に分ける
- 学習用データから、12項目と正解の関係をXGBoostに学ばせる
- 学習に使わなかった評価用400台で予測性能を確認する

XGBoostは、小さな決定木を順番に作り、前の木が間違えた部分を次の木で補いながら分類します。学習後のモデルは、12項目を受け取るとF1層がいる確率を0〜1で返します。

**実行**

次のコードセルを実行します。

**成功の目印**

- `学習データ: 1200台 / 調整データ: 400台 / 評価データ: 400台`
- `モデル: XGBoost`
- `評価完了` と評価結果の表

が表示されます。

In [ ]:
invalidate_from('evaluation')
require_data()
remaining_indices, test_indices = train_test_split(np.arange(len(known_pdf)), test_size=0.2, random_state=SPLIT_SEED, stratify=known_pdf.TARGET_F1)
train_indices, validation_indices = train_test_split(remaining_indices, test_size=0.25, random_state=SPLIT_SEED, stratify=known_pdf.TARGET_F1.iloc[remaining_indices])
train_pdf, validation_pdf, test_pdf = [known_pdf.iloc[indices].copy() for indices in (train_indices, validation_indices, test_indices)]
if [len(part) for part in (train_pdf, validation_pdf, test_pdf)] != [1200, 400, 400] or len(set(np.concatenate([train_indices, validation_indices, test_indices]))) != 2000:
    raise ValueError('学習・検証・最終テストの分割が不正です。')
train_features, train_labels = train_pdf[FEATURE_COLUMNS], train_pdf.TARGET_F1.astype('int64')
validation_features, validation_labels = validation_pdf[FEATURE_COLUMNS], validation_pdf.TARGET_F1.astype('int64')
model = XGBClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42,
    n_jobs=1,
)
model.fit(train_features, train_labels)
np.testing.assert_array_equal(model.classes_, [0, 1])
baseline = DummyClassifier(strategy='prior').fit(train_features, train_labels)
model_content_hash = model_digest(model)

def measure(part, estimator, split_name, model_label):
    target = part.TARGET_F1.astype('int64')
    probability = estimator.predict_proba(part[FEATURE_COLUMNS])[:, 1]
    if not np.isfinite(probability).all() or not ((probability >= 0) & (probability <= 1)).all():
        raise ValueError('評価確率が不正です。')
    predicted = (probability >= THRESHOLD).astype('int64')
    return {'SPLIT': split_name, 'MODEL': model_label, 'ROC_AUC': float(roc_auc_score(target, probability)), 'AVERAGE_PRECISION': float(average_precision_score(target, probability)), 'BRIER': float(brier_score_loss(target, probability)), 'PRECISION': float(precision_score(target, predicted, zero_division=0)), 'RECALL': float(recall_score(target, predicted, zero_division=0)), 'F1_SCORE': float(f1_score(target, predicted, zero_division=0)), 'PREVALENCE': float(target.mean()), 'CONFUSION': confusion_matrix(target, predicted, labels=[0, 1]).tolist()}

validation_rows = [measure(validation_pdf, estimator, 'VALIDATION', name) for name, estimator in [('Prior baseline', baseline), ('XGBoost', model)]]
importance_result = permutation_importance(model, validation_features, validation_labels, scoring='roc_auc', n_repeats=5, random_state=42, n_jobs=1)
importance_pdf = (
    pd.DataFrame({
        'FEATURE': FEATURE_COLUMNS,
        'IMPORTANCE': importance_result.importances_mean,
    })
    .sort_values('IMPORTANCE', ascending=True)
    .reset_index(drop=True)
)
freeze_token = digest([model_content_hash, THRESHOLD, frame_digest(known_pdf, ['DEVICE_ID'] + FEATURE_COLUMNS + ['TARGET_F1']), SPLIT_SEED, *[part.DEVICE_ID.tolist() for part in (train_pdf, validation_pdf, test_pdf)]])
ledger = globals().setdefault('_final_test_ledger', {})
ledger_key = DATASET_VERSION + ':' + EVALUATION_SCHEMA_VERSION
if ledger_key in ledger:
    if ledger[ledger_key]['freeze_token'] != freeze_token or 'rows' not in ledger[ledger_key]:
        raise RuntimeError('この最終テストは既に使用済みです。設定変更後の再採点は行いません。')
    final_test_record = ledger[ledger_key]
    print('同じ固定モデルの最終テスト記録を再表示します。再採点はしません。')
else:
    ledger[ledger_key] = {'freeze_token': freeze_token, 'started_at_utc': datetime.now(timezone.utc).isoformat(), 'model_hash': model_content_hash, 'threshold': THRESHOLD, 'dataset_version': DATASET_VERSION}
    test_rows = [measure(test_pdf, estimator, 'TEST', name) for name, estimator in [('Prior baseline', baseline), ('XGBoost', model)]]
    ledger[ledger_key].update(rows=test_rows, completed_at_utc=datetime.now(timezone.utc).isoformat())
    final_test_record = ledger[ledger_key]
metrics_df = pd.DataFrame(validation_rows + final_test_record['rows'])
print('学習データ:', len(train_pdf), '台 / 調整データ:', len(validation_pdf), '台 / 評価データ:', len(test_pdf), '台')
print('モデル: XGBoost')
print('評価完了')
learner_metrics = (
    metrics_df[metrics_df['MODEL'].eq('XGBoost')]
    [['SPLIT', 'ROC_AUC', 'PRECISION', 'RECALL']]
    .rename(columns={'SPLIT': 'データ', 'ROC_AUC': 'ROC AUC', 'PRECISION': '適合率', 'RECALL': '再現率'})
    .round(3)
)
display(learner_metrics)
if model_digest(model) != model_content_hash:
    raise ValueError('評価中にモデル内容が変わりました。登録を停止します。')
evaluation_token = evaluated_state()

## 6. モデル全体の判断材料を見る

**このセルで行うこと**

調整用400台で各項目を入れ替え、予測性能がどれだけ下がるかを比べます。大きく下がる項目ほど、モデル全体が強く使っている判断材料です。

この重要度は、予測に使われた度合いを示します。原因や因果関係を証明するものではありません。

**実行**

次のコードセルを実行します。

**成功の目印**

`モデル全体の特徴量重要度` という横棒グラフが表示されます。

In [ ]:
require_evaluation()

importance_fig = px.bar(
    importance_pdf,
    x='IMPORTANCE',
    y='FEATURE',
    orientation='h',
    title='モデル全体の特徴量重要度',
    labels={
        'IMPORTANCE': '調整用ROC AUCの平均変化',
        'FEATURE': '視聴項目',
    },
    template='plotly_white',
)
importance_fig.update_traces(marker_color='#29B5E8')
importance_fig.add_vline(x=0, line_color='#7F8C8D', line_dash='dash')
importance_fig.show()
positive_importance = importance_pdf[importance_pdf['IMPORTANCE'] > 0]
if positive_importance.empty:
    print('性能低下が確認できた項目はありませんでした。')
else:
    print('最も重要な項目:', positive_importance.iloc[-1]['FEATURE'])

## 7. モデルをSnowflakeへ登録する

**このセルで行うこと**

学習済みのXGBoostモデルを、`TV_F1_PRESENCE_MODEL` の初期版`V3`としてModel Registryへ登録します。V3が既存の場合は、Notebook先頭で設定した未使用版を使います。学習データの一部も説明用の基準として一緒に登録します。

Model Registryでは、モデル本体だけでなく、版、評価指標、実行環境、予測関数、説明関数をSnowflake上でまとめて管理できます。

**実行**

次のコードセルを実行します。

**成功の目印**

`モデル登録完了: TV_F1_PRESENCE_MODEL （選択した未使用版）` と、`predict_proba`、`explain` が表示されます。

In [ ]:
invalidate_from('registration')
require_evaluation()
registry = Registry(session=session, database_name='BCAST_PLATFORM_HANDSON', schema_name='ML')
existing_models = registry.show_models()
existing_models.columns = [str(column).upper() for column in existing_models.columns]
if not existing_models.empty and 'NAME' not in existing_models.columns:
    raise RuntimeError('モデル一覧の列が不明です。安全のため停止します。')
if not existing_models.empty and existing_models['NAME'].astype('string').str.upper().eq(MODEL_NAME).any():
    existing_versions = registry.get_model(MODEL_NAME).show_versions()
    existing_versions.columns = [str(column).upper() for column in existing_versions.columns]
    if 'NAME' not in existing_versions.columns:
        raise RuntimeError('モデル版一覧の列が不明です。安全のため停止します。')
    if existing_versions['NAME'].astype('string').str.upper().eq(MODEL_VERSION).any():
        raise RuntimeError('同じ版が存在します。既存版は削除せず、MODEL_VERSIONを未使用のV4などへ変更して先頭から実行してください。')
np.testing.assert_array_equal(model.classes_, [0, 1])
probability_signature = model_signature.infer_signature(train_pdf[FEATURE_COLUMNS].head(10), model.predict_proba(train_pdf[FEATURE_COLUMNS].head(10)), output_feature_names=['PROB_NO_F1', 'PROB_F1'])
registered_metrics = {str(row.SPLIT).lower() + '_' + column.lower(): float(getattr(row, column)) for row in metrics_df.itertuples() if row.MODEL == 'XGBoost' for column in ['ROC_AUC', 'AVERAGE_PRECISION', 'BRIER', 'PRECISION', 'RECALL', 'F1_SCORE']}
candidate_version = registry.log_model(
    model,
    model_name=MODEL_NAME,
    version_name=MODEL_VERSION,
    signatures={'predict_proba': probability_signature},
    sample_input_data=train_pdf[FEATURE_COLUMNS].head(200),
    conda_dependencies=['xgboost==' + xgboost.__version__, 'scikit-learn==' + sklearn.__version__],
    target_platforms=['WAREHOUSE'],
    python_version='3.12',
    options={'relax_version': False, 'enable_explainability': True},
    metrics=registered_metrics,
    comment=json.dumps({'dataset_version': DATASET_VERSION, 'threshold': THRESHOLD, 'algorithm': 'XGBoost', 'model_hash': model_content_hash, 'release_hash': release_fingerprint, 'evaluation_token': evaluation_token, 'semantics': 'Synthetic household F1 presence; not current viewer or person count.'}, sort_keys=True)
)
require_evaluation()
if (candidate_version.model_name, candidate_version.version_name) != (MODEL_NAME, MODEL_VERSION):
    raise ValueError('登録先モデル版が一致しません。後続を停止します。')
registered_version = candidate_version
registered_name, registered_version_name = registered_version.model_name, registered_version.version_name
registered_functions = registered_version.show_functions()
function_names = {str(function.name).lower() for function in registered_functions}
if not {'predict_proba', 'explain'}.issubset(function_names):
    globals().pop('registered_version', None)
    globals().pop('registered_name', None)
    globals().pop('registered_version_name', None)
    raise RuntimeError('モデル版は登録されましたが、predict_probaとexplainを確認できません。既存版は削除せず、MODEL_VERSIONを未使用版へ変更して先頭から実行してください。')
registration_token = digest([evaluation_token, registered_name, registered_version_name])
print('モデル登録完了:', registered_name, registered_version_name)
registered_functions

## 8. 登録モデルの判断理由をSnowflakeで説明する

**このセルで行うこと**

Model Registryへ登録された `explain` 関数を、調整用データ100台に対してSnowflake上で実行します。

- 全体表示: 各項目が予測を動かした大きさの平均
- 1台の表示: 各項目が「F1層あり」の方向、または「なし」の方向へどれだけ働いたか

SHAP値はモデルの判断理由を説明する値です。プラスは「F1層あり」の方向、マイナスは「なし」の方向を表します。値は確率の増減そのものではなく、XGBoost内部の生のモデルスコアへの寄与です。因果関係を示すものではありません。

**成功の目印**

`登録モデルのSHAP重要度` と `1台の予測を動かした項目` の2つのグラフが表示されます。

In [ ]:
require_registration()

explain_input_pdf = validation_pdf[FEATURE_COLUMNS].head(100).reset_index(drop=True)
explain_schema = StructType([StructField(column, DoubleType()) for column in FEATURE_COLUMNS])
explain_input_sdf = session.create_dataframe(
    explain_input_pdf.to_numpy().tolist(),
    schema=explain_schema,
)
explain_output = (
    registered_version
    .run(explain_input_sdf, function_name='explain')
    .limit(101)
    .to_pandas()
)
explain_output.columns = [str(column).upper() for column in explain_output.columns]
explanation_columns = [column for column in explain_output.columns if column.endswith('_EXPLANATION')]
expected_explanation_columns = {column + '_EXPLANATION' for column in FEATURE_COLUMNS}
if len(explain_output) != 100 or set(explanation_columns) != expected_explanation_columns:
    raise RuntimeError('SHAP説明の行数または列が登録モデルと一致しません。')

shap_pdf = explain_output[explanation_columns].astype('float64').rename(
    columns=lambda column: column.removesuffix('_EXPLANATION')
)
if not np.isfinite(shap_pdf.to_numpy()).all():
    raise ValueError('SHAP値に欠損または無限大があります。')

shap_importance_pdf = (
    shap_pdf.abs().mean()
    .rename('MEAN_ABS_SHAP')
    .rename_axis('FEATURE')
    .reset_index()
    .sort_values('MEAN_ABS_SHAP', ascending=True)
)
shap_fig = px.bar(
    shap_importance_pdf,
    x='MEAN_ABS_SHAP',
    y='FEATURE',
    orientation='h',
    title='登録モデルのSHAP重要度（調整用100台）',
    labels={
        'MEAN_ABS_SHAP': '生のモデルスコアを動かした大きさの平均',
        'FEATURE': '視聴項目',
    },
    template='plotly_white',
)
shap_fig.update_traces(marker_color='#29B5E8')
shap_fig.show()

example_position = int(np.argmax(model.predict_proba(explain_input_pdf)[:, 1]))
example_device_id = validation_pdf.iloc[example_position]['DEVICE_ID']
example_input_pdf = explain_input_pdf.iloc[[example_position]].reset_index(drop=True)
example_input_sdf = session.create_dataframe(
    example_input_pdf.to_numpy().tolist(),
    schema=explain_schema,
)
example_output = (
    registered_version
    .run(example_input_sdf, function_name='explain')
    .limit(2)
    .to_pandas()
)
example_output.columns = [str(column).upper() for column in example_output.columns]
if len(example_output) != 1 or set(column for column in example_output.columns if column.endswith('_EXPLANATION')) != expected_explanation_columns:
    raise RuntimeError('1台分のSHAP説明が登録モデルと一致しません。')
example_shap_pdf = (
    example_output[explanation_columns]
    .astype('float64')
    .rename(columns=lambda column: column.removesuffix('_EXPLANATION'))
    .iloc[0]
    .rename('SHAP_VALUE')
    .rename_axis('FEATURE')
    .reset_index()
    .sort_values('SHAP_VALUE')
)
example_shap_pdf['DIRECTION'] = np.where(
    example_shap_pdf['SHAP_VALUE'] >= 0,
    'F1層ありの方向',
    'F1層なしの方向',
)
example_fig = px.bar(
    example_shap_pdf,
    x='SHAP_VALUE',
    y='FEATURE',
    color='DIRECTION',
    orientation='h',
    title=f'1台の予測を動かした項目: {example_device_id}',
    labels={
        'SHAP_VALUE': '生のモデルスコアへの寄与',
        'FEATURE': '視聴項目',
        'DIRECTION': '方向',
    },
    color_discrete_map={
        'F1層ありの方向': '#29B5E8',
        'F1層なしの方向': '#7F8C8D',
    },
    template='plotly_white',
)
example_fig.show()
print('Snowflake上の登録モデルでSHAP説明完了:', len(shap_pdf), '台')

## 9. 登録したモデルで20,000台を予測する

**入力**

20,000台それぞれの12項目の視聴傾向を、Model Registryへ登録したモデルに渡します。

**出力**

- `PROB_F1`: F1層がいる確率（0〜1）
- `PREDICTED_HAS_F1`: 確率が0.50以上なら1、未満なら0

**実行**

次のコードセルを実行します。

**成功の目印**

`予測完了: 20000台` と、`DEVICE_ID`・確率・判定の先頭5行が表示されます。

In [ ]:
invalidate_from('prediction')
require_registration()
prediction_input = features_pdf[FEATURE_COLUMNS].drop_duplicates().reset_index(drop=True)
prediction_schema = StructType([StructField(column, DoubleType()) for column in FEATURE_COLUMNS])
prediction_sdf = session.create_dataframe(prediction_input.to_numpy().tolist(), schema=prediction_schema)
registry_output = registered_version.run(prediction_sdf, function_name='predict_proba').limit(20001).to_pandas()
if not isinstance(registry_output, pd.DataFrame):
    raise RuntimeError('推論結果を表へ変換できません。')
registry_output.columns = [str(column).upper() for column in registry_output.columns]
expected_columns = set(FEATURE_COLUMNS) | {'PROB_NO_F1', 'PROB_F1'}
if not registry_output.columns.is_unique or set(registry_output.columns) != expected_columns:
    raise RuntimeError('推論列が登録した署名と一致しません。列名を推測して保存しません。')
if len(registry_output) != len(prediction_input) or registry_output.duplicated(FEATURE_COLUMNS).any():
    raise ValueError('推論行数または特徴量の一意性が一致しません。')
probabilities = registry_output[['PROB_NO_F1', 'PROB_F1']].astype('float64')
if not np.isfinite(probabilities.to_numpy()).all() or not probabilities.ge(0).all().all() or not probabilities.le(1).all().all() or not np.allclose(probabilities.sum(axis=1), 1, rtol=0, atol=1e-12):
    raise ValueError('確率は有限の0〜1で、両クラスの合計が1である必要があります。')
candidate_predictions = features_pdf.merge(registry_output, on=FEATURE_COLUMNS, how='left', validate='many_to_one')
if len(candidate_predictions) != 20000 or candidate_predictions.PROB_F1.isna().any():
    raise ValueError('全TVへの予測対応を確認できません。')
if not np.allclose(candidate_predictions.PROB_F1, model.predict_proba(candidate_predictions[FEATURE_COLUMNS])[:, 1], rtol=1e-8, atol=1e-10):
    raise ValueError('登録モデルの確率が評価済みモデルと一致しません。')
candidate_predictions['PREDICTED_HAS_F1'] = (candidate_predictions.PROB_F1 >= THRESHOLD).astype('int64')
require_registration()
prediction_pdf = candidate_predictions
prediction_token = digest([registration_token, THRESHOLD, DATASET_VERSION, frame_digest(prediction_pdf, ['DEVICE_ID', 'PROB_F1', 'PREDICTED_HAS_F1'])])
print('予測完了:', len(prediction_pdf), '台')
prediction_pdf[['DEVICE_ID', 'PROB_F1', 'PREDICTED_HAS_F1']].head()

## 10. 予測結果をSnowflakeへ保存する

**このセルで行うこと**

20,000台の端末ID・確率・判定・モデル版を、テーブル `BCAST_PLATFORM_HANDSON.ML.PREDICTIONS` に保存します。

**実行**

1. 最初は `SAVE_RESULTS = False` のまま次のコードセルを実行します。
2. 保存確認のメッセージが表示されたら、`SAVE_RESULTS = True` に変更します。
3. 同じコードセルをもう一度実行します。

**成功の目印**

`保存完了: BCAST_PLATFORM_HANDSON.ML.PREDICTIONS 20000行` と表示されます。

In [ ]:
SAVE_RESULTS = False
globals().pop('publication_token', None)
globals().pop('published_table', None)
if not SAVE_RESULTS:
    raise RuntimeError('保存する場合はSAVE_RESULTSをTrueに変更してください。')
if 'prediction_token' not in globals():
    raise ValueError('保存前確認: このモデル版で推論を完了してください。')
require_registration()
if prediction_token != digest([registration_token, THRESHOLD, DATASET_VERSION, frame_digest(prediction_pdf, ['DEVICE_ID', 'PROB_F1', 'PREDICTED_HAS_F1'])]):
    raise ValueError('保存前確認: 予測・モデル・閾値・データ版の来歴が一致しません。')
if len(prediction_pdf) != 20000 or prediction_pdf.DEVICE_ID.isna().any() or not prediction_pdf.DEVICE_ID.is_unique or set(prediction_pdf.DEVICE_ID) != EXPECTED_DEVICES:
    raise ValueError('保存前確認: 全20000台のIDが一致しません。')
probability = prediction_pdf.PROB_F1.astype('float64')
if not np.isfinite(probability).all() or not probability.between(0, 1).all() or not prediction_pdf.PREDICTED_HAS_F1.eq((probability >= THRESHOLD).astype('int64')).all():
    raise ValueError('保存前確認: 確率と固定閾値によるラベルが一致しません。')
output_rows = [(str(row.DEVICE_ID), float(row.PROB_F1), int(row.PREDICTED_HAS_F1)) for row in prediction_pdf.itertuples(index=False)]
output_schema = StructType([StructField('DEVICE_ID', StringType()), StructField('PROB_F1', DoubleType()), StructField('PREDICTED_HAS_F1', LongType())])
output_sdf = session.create_dataframe(output_rows, schema=output_schema).with_column('MODEL_NAME', sf.lit(registered_name)).with_column('MODEL_VERSION', sf.lit(registered_version_name)).with_column('PREDICTED_AT', sf.convert_timezone(sf.lit('UTC'), sf.current_timestamp()).cast('timestamp_ntz')).with_column('PREDICTION_THRESHOLD', sf.lit(float(THRESHOLD)).cast('double')).with_column('DATASET_VERSION', sf.lit(DATASET_VERSION))
staging_table = 'BCAST_PLATFORM_HANDSON.ML.PREDICTIONS_STAGE_' + uuid.uuid4().hex.upper()
output_sdf.write.mode('errorifexists').save_as_table(staging_table, table_type='temporary')
staged_sdf = session.table(staging_table)
saved_pdf = staged_sdf.limit(20001).to_pandas()
save_columns = ['DEVICE_ID', 'PROB_F1', 'PREDICTED_HAS_F1', 'MODEL_NAME', 'MODEL_VERSION', 'PREDICTED_AT', 'PREDICTION_THRESHOLD', 'DATASET_VERSION']
if list(saved_pdf.columns) != save_columns or saved_pdf.isna().any().any() or len(saved_pdf) != 20000 or not saved_pdf.DEVICE_ID.is_unique or set(saved_pdf.DEVICE_ID) != EXPECTED_DEVICES:
    raise RuntimeError('ステージングの列・端末照合に失敗しました。既存予測は変更していません。')
fields = {field.name: field.datatype for field in staged_sdf.schema.fields}
if not isinstance(fields['PREDICTION_THRESHOLD'], DoubleType) or not isinstance(fields['PROB_F1'], DoubleType) or getattr(getattr(fields['PREDICTED_AT'], 'tz', None), 'name', None) != 'NTZ':
    raise RuntimeError('ステージングの確率・閾値・日時型が不正です。既存予測は変更していません。')
compare_columns = ['DEVICE_ID', 'PROB_F1', 'PREDICTED_HAS_F1']
pd.testing.assert_frame_equal(prediction_pdf[compare_columns].sort_values('DEVICE_ID').reset_index(drop=True), saved_pdf[compare_columns].sort_values('DEVICE_ID').reset_index(drop=True), check_dtype=False, check_exact=True)
if not saved_pdf.MODEL_NAME.eq(registered_name).all() or not saved_pdf.MODEL_VERSION.eq(registered_version_name).all() or not saved_pdf.PREDICTION_THRESHOLD.eq(THRESHOLD).all() or not saved_pdf.DATASET_VERSION.eq(DATASET_VERSION).all():
    raise RuntimeError('ステージングのモデル版・閾値・データ版が不正です。既存予測は変更していません。')
require_registration()
if prediction_token != digest([registration_token, THRESHOLD, DATASET_VERSION, frame_digest(prediction_pdf, compare_columns)]):
    raise ValueError('公開直前に予測が変わりました。既存予測は変更していません。')
if not re.fullmatch(r'BCAST_PLATFORM_HANDSON\.ML\.PREDICTIONS_STAGE_[0-9A-F]{32}', staging_table):
    raise ValueError('ステージング識別子が不正です。')
session.sql(f"CREATE OR REPLACE TABLE BCAST_PLATFORM_HANDSON.ML.PREDICTIONS COPY GRANTS AS SELECT {', '.join(save_columns)} FROM {staging_table}").collect()
published_table = 'BCAST_PLATFORM_HANDSON.ML.PREDICTIONS'
publication_token = prediction_token
print('保存完了: BCAST_PLATFORM_HANDSON.ML.PREDICTIONS', len(saved_pdf), '行')

## 11. Notebookサービスを停止する

データの観察、モデルの学習・評価、Model Registryへの登録、20,000台の予測、結果保存まで完了しました。

最後に **Connected → サービス名 → Suspend** を選び、状態が **SUSPENDED** になったことを確認します。